In [16]:
# Task 1.1

import csv
import json

def parse_row(row):
    return {
        **row,
        "age": int(row["age"]),
        "fee_inr": float(row["fee_inr"]),
        "prior_no_shows": int(row["prior_no_shows"]),
        "distance_km": float(row["distance_km"]),
    }

accepted = []
rejected = []


with open("appointments.csv", newline="", encoding="utf-8") as f:
  reader = csv.DictReader(f)

  headers = reader.fieldnames
  print("Headers:", headers)

  row_count = 0

  for row in reader:
    row_count += 1

    try:
      parsed = parse_row(row)
      accepted.append(parsed)
    except ValueError as e:
      row["_error"] = str(e)
      rejected.append(row)

print("Row count:", row_count)
print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

with open("rejected_rows.json", "w", encoding="utf-8") as f:
    json.dump(rejected, f, indent=2)

Headers: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Row count: 1020
Accepted: 988
Rejected: 32


In [17]:
# Task 1.2

import pandas as pd

# Load data
df = pd.read_csv("appointments.csv")

# Initial inspection
print("Shape:")
print(df.shape)

print("\nInfo:")
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())



Shape:
(1020, 14)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB
None

Missing values:
appointment_id      0
patient_id          0
c

In [18]:

# ---------------------------
# Step (a): Drop exact duplicates
# ---------------------------
df = df.drop_duplicates()
print("\nAfter dropping duplicate rows:", df.shape)


After dropping duplicate rows: (1000, 14)


In [19]:
# ---------------------------
# Step (b): Clean booking_channel
# ---------------------------
df["booking_channel"] = (
    df["booking_channel"]
      .str.strip()
      .str.lower()
      .replace({
          "app": "App",
          "phone": "Phone",
          "walk-in": "Walk-In",
          "walk in": "Walk-In",
          "walkin": "Walk-In"
      })
)

print("\nAfter cleaning booking_channel:", df.shape)


After cleaning booking_channel: (1000, 14)


In [20]:
# ---------------------------
# Step (c): Convert appt_date
# ---------------------------
df["appt_date"] = pd.to_datetime(
    df["appt_date"],
    errors="coerce"
)

print("\nAfter converting appt_date:", df.shape)


After converting appt_date: (1000, 14)


In [21]:
# ---------------------------
# Step (d): Convert booked_on
# ---------------------------
df["booked_on"] = pd.to_datetime(
    df["booked_on"],
    errors="coerce"
)

print("\nAfter converting booked_on:", df.shape)


After converting booked_on: (1000, 14)


In [22]:
# ---------------------------
# Step (e): Drop rows where
# appt_date is missing after conversion
# OR status is missing
# ---------------------------
df = df.dropna(
    subset=["appt_date", "status"]
)

print("\nAfter dropping missing appt_date/status:", df.shape)

print("\nFinal shape:", df.shape)


After dropping missing appt_date/status: (990, 14)

Final shape: (990, 14)


In [23]:
# Task 1.3

import numpy as np
import pandas as pd

# ---------------------------
# (a) Invalid age -> missing
# ---------------------------
df.loc[(df["age"] < 1) | (df["age"] > 100), "age"] = np.nan

In [24]:
# ---------------------------
# (b) Fill gaps
# ---------------------------

# age -> median, then convert to int
df["age"] = (
    df["age"]
    .fillna(df["age"].median())
    .round()
    .astype(int)
)

In [25]:
# distance_km -> median
df["distance_km"] = df["distance_km"].fillna(
    df["distance_km"].median()
)

In [26]:
# ---------------------------
# (c) Outlier capping (IQR)
# ---------------------------

def cap_outliers(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    before = df[col].copy()

    df[col] = np.clip(df[col], lower, upper)

    capped_count = (before != df[col]).sum()

    return capped_count

distance_capped = cap_outliers(df, "distance_km")
fee_capped = cap_outliers(df, "fee_inr")

In [27]:
# ---------------------------
# (d) Rounding
# ---------------------------

df["distance_km"] = df["distance_km"].round(1)

df["fee_inr"] = (
    df["fee_inr"]
    .round()
    .astype(int)
)

In [28]:
# ---------------------------
# Required output
# ---------------------------

print("Distance values capped:", distance_capped)
print("Fee values capped:", fee_capped)

Distance values capped: 26
Fee values capped: 3


In [29]:
# Task 1.4

# ---------------------------
# Feature engineering
# ---------------------------

df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

df["no_show"] = (df["status"] == "No-Show").astype(int)

# ---------------------------
# NumPy matrix
# ---------------------------

X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()

# ---------------------------
# (a) Mean of each column
# ---------------------------

column_means = X.mean(axis=0)

print("Age mean:", column_means[0])
print("Distance mean:", column_means[1])
print("Fee mean:", column_means[2])
print("Lead-days mean:", column_means[3])

# ---------------------------
# (b) Rows with lead_days > 14
# ---------------------------

mask = X[:, 3] > 14

count_gt14 = mask.sum()
share_gt14 = mask.mean()

print("Rows with lead_days > 14:", count_gt14)
print("Share with lead_days > 14:", share_gt14)

# ---------------------------
# (c) Z-scores for distance_km
# ---------------------------

distance = X[:, 1]

z_scores = (distance - distance.mean()) / distance.std()

outlier_count = (np.abs(z_scores) > 2).sum()

print("Rows with |z| > 2:", outlier_count)

Age mean: 48.173737373737374
Distance mean: 9.50090909090909
Fee mean: 635.7141414141414
Lead-days mean: 11.819191919191919
Rows with lead_days > 14: 296
Share with lead_days > 14: 0.298989898989899
Rows with |z| > 2: 53


In [30]:
# Task 1.5

import pandas as pd
import requests

clinics = pd.read_csv("clinics.csv")

temps = []

for _, row in clinics.iterrows():

    temp = None

    for attempt in range(2):
        try:
            r = requests.get(
                "https://api.open-meteo.com/v1/forecast",
                params={
                    "latitude": row["latitude"],
                    "longitude": row["longitude"],
                    "hourly": "temperature_2m",
                    "forecast_days": 1
                },
                timeout=10
            )

            if r.status_code == 200:

                data = r.json()

                temp = data["hourly"]["temperature_2m"][0]
                break

            else:
                raise Exception(f"status {r.status_code}")

        except requests.exceptions.RequestException as e:

            if attempt == 1:
                print(f"{row['clinic_name']}: {e}")

    temps.append(temp)

clinics["temp_c"] = temps

clinics.to_csv(
    "clinics_enriched.csv",
    index=False
)

In [31]:
# Task 1.6

import pandas as pd

# -------------------------
# Check available columns
# -------------------------
print("Columns:")
print(df.columns.tolist())


Columns:
['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status', 'lead_days', 'no_show']


In [32]:
# -------------------------
# Convert dates if present
# -------------------------
for col in ["booked_on", "appt_date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col]).dt.strftime("%Y-%m-%d")

In [33]:
# -------------------------
# Required output columns
# -------------------------
required_cols = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

missing_cols = [c for c in required_cols if c not in df.columns]

if missing_cols:
    print("Missing columns:", missing_cols)
else:

    df_export = df[required_cols].copy()

    print("Rows:", len(df_export))
    print("Columns:", len(df_export.columns))

    print(
        "Distinct appointment_id:",
        df_export["appointment_id"].nunique()
    )

    print(
        "Total missing:",
        df_export.isnull().sum().sum()
    )

    df_export.to_csv(
        "appointments_clean.csv",
        index=False
    )

    print("appointments_clean.csv created")


Rows: 990
Columns: 16
Distinct appointment_id: 990
Total missing: 0
appointments_clean.csv created


In [34]:
# -------------------------
# SQL generation
# -------------------------

try:
    clinics = pd.read_csv("clinics_enriched.csv")

    insert_count = 0

    with open("load_data.sql", "w", encoding="utf-8") as f:

        # Clinics first
        for _, row in clinics.iterrows():

            temp_c = (
                "NULL"
                if pd.isna(row["temp_c"])
                else row["temp_c"]
            )

            f.write(
                f"INSERT INTO clinics VALUES "
                f"('{row['clinic_id']}',"
                f"'{row['clinic_name']}',"
                f"{temp_c});\n"
            )

            insert_count += 1

        # Appointments second
        if not missing_cols:

            for _, row in df_export.iterrows():

                f.write(
                    f"INSERT INTO appointments VALUES ("
                    f"'{row['appointment_id']}',"
                    f"'{row['patient_id']}',"
                    f"'{row['clinic_id']}',"
                    f"'{row['booked_on']}',"
                    f"'{row['appt_date']}',"
                    f"{row['age']},"
                    f"'{row['gender']}',"
                    f"{row['distance_km']},"
                    f"{row['fee_inr']},"
                    f"'{row['booking_channel']}',"
                    f"'{row['visit_type']}',"
                    f"{row['reminder_sent']},"
                    f"{row['prior_no_shows']},"
                    f"'{row['status']}',"
                    f"{row['lead_days']},"
                    f"{row['no_show']}"
                    f");\n"
                )

                insert_count += 1

    print("INSERT statements:", insert_count)

except FileNotFoundError:
    print("clinics_enriched.csv not found")

INSERT statements: 998


In [35]:
# Task3.1

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load cleaned file from Part 1
df = pd.read_csv("appointments_clean.csv")

In [36]:
# -----------------------------------
# Convert reminder_sent Yes/No -> 1/0
# -----------------------------------

df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

In [37]:
# -----------------------------------
# Features
# -----------------------------------

X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
]

# One-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

In [38]:
# Target
y = df["no_show"]

# -----------------------------------
# Train/Test Split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [39]:
# -----------------------------------
# Required checks
# -----------------------------------

print("X shape:", X.shape)
print("Number of features:", X.shape[1])

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))

print("No-shows in test set:", y_test.sum())

# -----------------------------------
# Baseline model
# Always predict attended (0)
# -----------------------------------

baseline_pred = [0] * len(y_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_pred
)

print("Baseline accuracy:", baseline_accuracy)

X shape: (990, 11)
Number of features: 11
Train rows: 792
Test rows: 198
No-shows in test set: 49
Baseline accuracy: 0.7525252525252525


In [40]:
# Task3.2

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [41]:
# ---------------------------
# Scale features
# ---------------------------

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [42]:
# ---------------------------
# Logistic Regression
# ---------------------------

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_s, y_train)


LogisticRegression(max_iter=1000, random_state=42)

In [43]:
# ---------------------------
# Predictions
# ---------------------------

y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]

In [44]:
# ---------------------------
# Metrics
# ---------------------------

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

cm = confusion_matrix(y_test, y_pred)

print("Accuracy :", acc)
print("Precision:", prec)
print("Recall   :", rec)
print("F1       :", f1)
print("ROC-AUC  :", auc)

print("\nConfusion Matrix")
print(cm)

Accuracy : 0.7878787878787878
Precision: 0.6296296296296297
Recall   : 0.3469387755102041
F1       : 0.4473684210526316
ROC-AUC  : 0.788933022873579

Confusion Matrix
[[139  10]
 [ 32  17]]


In [45]:
# ---------------------------
# Coefficients
# ---------------------------

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

coef_df = coef_df.sort_values(
    "coefficient",
    ascending=False
)

print("\nCoefficients")
print(coef_df)


Coefficients
                    feature  coefficient
4            prior_no_shows     0.649612
3                 lead_days     0.614471
2                   fee_inr     0.452761
1               distance_km     0.253448
7     booking_channel_Phone     0.215881
9   visit_type_Consultation     0.015658
10     visit_type_Follow-up    -0.015658
6       booking_channel_App    -0.046804
8   booking_channel_Walk-In    -0.186364
0                       age    -0.253880
5             reminder_sent    -0.580669


In [46]:
# ---------------------------
# Reminder coefficient
# ---------------------------

idx = list(X.columns).index("reminder_sent")

reminder_coef = model.coef_[0][idx]

print("\nReminder coefficient:")
print(reminder_coef)

print("\nexp(coef)")
print(np.exp(reminder_coef))


Reminder coefficient:
-0.5806686033659922

exp(coef)
0.559524141750829


In [47]:
# ---------------------------
# Convert back to real-world
# No -> Yes effect
# ---------------------------

scale = scaler.scale_[idx]

print("\nStandard deviation:")
print(scale)

real_effect = np.exp(
    reminder_coef / scale
)

print("\nexp(coef / scale)")
print(real_effect)


Standard deviation:
0.4949752621616814

exp(coef / scale)
0.3093980903124433


In [48]:
# Task3.3

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# =====================================
# Model 1: Full Depth Tree
# =====================================

tree_full = DecisionTreeClassifier(
    random_state=42
)

tree_full.fit(X_train, y_train)

train_pred_full = tree_full.predict(X_train)
test_pred_full = tree_full.predict(X_test)

train_acc_full = accuracy_score(y_train, train_pred_full)
test_acc_full = accuracy_score(y_test, test_pred_full)

print("FULL TREE")
print("Depth:", tree_full.get_depth())
print("Train Accuracy:", round(train_acc_full, 4))
print("Test Accuracy :", round(test_acc_full, 4))

FULL TREE
Depth: 15
Train Accuracy: 1.0
Test Accuracy : 0.6717


In [49]:
# =====================================
# Model 2: Depth = 3
# =====================================

tree_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

tree_3.fit(X_train, y_train)

train_pred_3 = tree_3.predict(X_train)
test_pred_3 = tree_3.predict(X_test)

train_acc_3 = accuracy_score(y_train, train_pred_3)
test_acc_3 = accuracy_score(y_test, test_pred_3)

print("\nDEPTH-3 TREE")
print("Depth:", tree_3.get_depth())
print("Train Accuracy:", round(train_acc_3, 4))
print("Test Accuracy :", round(test_acc_3, 4))


DEPTH-3 TREE
Depth: 3
Train Accuracy: 0.822
Test Accuracy : 0.7778


In [50]:
# Task3.4

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score
)

# =============================
# Random Forest
# =============================

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

# Predictions
y_train_pred_rf = rf.predict(X_train)
y_test_pred_rf = rf.predict(X_test)

y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Metrics
train_acc_rf = accuracy_score(y_train, y_train_pred_rf)
test_acc_rf = accuracy_score(y_test, y_test_pred_rf)

recall_rf = recall_score(y_test, y_test_pred_rf)

auc_rf = roc_auc_score(y_test, y_prob_rf)

print("Random Forest Results")
print("---------------------")
print("Training Accuracy :", train_acc_rf)
print("Test Accuracy     :", test_acc_rf)
print("Recall            :", recall_rf)
print("ROC-AUC           :", auc_rf)

Random Forest Results
---------------------
Training Accuracy : 1.0
Test Accuracy     : 0.7474747474747475
Recall            : 0.30612244897959184
ROC-AUC           : 0.7057252431173813


In [51]:
# =============================
# Feature Importances
# =============================

importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
)

print("\nTop 3 Features")
print(importance_df.head(3))


Top 3 Features
       feature  importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


In [52]:
# Task 3.5

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import (
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [53]:
# ==================================================
# A) Tune Random Forest
# ==================================================

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best Params:")
print(rf_grid.best_params_)

print("\nBest CV ROC-AUC:")
print(rf_grid.best_score_)

Best Params:
{'max_depth': 3, 'n_estimators': 200}

Best CV ROC-AUC:
0.7673242251739405


In [54]:
# ==================================================
# B) Compare models using training CV ROC-AUC only
# ==================================================

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

tree3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

rf_best = rf_grid.best_estimator_

lr_auc = cross_val_score(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

tree_auc = cross_val_score(
    tree3,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

rf_auc = cross_val_score(
    rf_best,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

print("\nTraining CV ROC-AUC")
print("Logistic Regression:", lr_auc)
print("Depth-3 Tree:", tree_auc)
print("Random Forest:", rf_auc)


Training CV ROC-AUC
Logistic Regression: 0.7737139899948249
Depth-3 Tree: 0.6784494968661952
Random Forest: 0.7673242251739405


In [55]:
# ==================================================
# C) Threshold selection
# ==================================================

oof_probs = cross_val_predict(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

for threshold in [0.50, 0.35]:

    preds = (oof_probs >= threshold).astype(int)

    precision = precision_score(
        y_train,
        preds
    )

    recall = recall_score(
        y_train,
        preds
    )

    f1 = f1_score(
        y_train,
        preds
    )

    flagged = preds.sum()

    print("\nThreshold =", threshold)
    print("Precision =", precision)
    print("Recall =", recall)
    print("F1 =", f1)
    print("Flagged =", flagged)


Threshold = 0.5
Precision = 0.5753424657534246
Recall = 0.2485207100591716
F1 = 0.34710743801652894
Flagged = 73

Threshold = 0.35
Precision = 0.512987012987013
Recall = 0.46745562130177515
F1 = 0.4891640866873065
Flagged = 154


In [56]:
# ==================================================
# D) Choose threshold 0.35
# ==================================================

threshold = 0.35

# ==================================================
# E) Fit final model on all training data
# ==================================================

lr_pipe.fit(
    X_train,
    y_train
)

test_probs = lr_pipe.predict_proba(
    X_test
)[:, 1]

test_pred = (
    test_probs >= threshold
).astype(int)

In [57]:
# ==================================================
# F) Test-set evaluation
# ==================================================

precision = precision_score(
    y_test,
    test_pred
)

recall = recall_score(
    y_test,
    test_pred
)

roc_auc = roc_auc_score(
    y_test,
    test_probs
)

flagged = test_pred.sum()

print("\nTEST RESULTS")
print("Precision:", precision)
print("Recall:", recall)
print("ROC-AUC:", roc_auc)
print("Flagged:", flagged)


TEST RESULTS
Precision: 0.5714285714285714
Recall: 0.5714285714285714
ROC-AUC: 0.788933022873579
Flagged: 49


In [58]:
# ==================================================
# G) Risk score file
# ==================================================

risk_scores = pd.DataFrame({
    "appointment_id":
        df.loc[X_test.index,
               "appointment_id"],
    "no_show":
        y_test,
    "risk_prob":
        test_probs,
    "flag":
        test_pred
})

risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print(
    "\ntest_risk_scores.csv written"
)


test_risk_scores.csv written


In [62]:
pip install google-genai python-dotenv

In [67]:
!pip install -q google-genai

In [ ]:
# Task4.1
import json

from google import genai
from google.genai import types

API_KEY = input("Paste your Gemini API key: ")

client = genai.Client(api_key=API_KEY)

MODEL_NAME = "gemini-3.8-flash"

In [75]:
SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {"type": "string"},
        "confidence": {"type": "number"},
        "summary": {"type": "string"},
        "recommended_action": {"type": "string"}
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You analyse clinic no-show notes.

Return JSON only.

root_cause must be one of:

forgot_or_no_reminder
transport_problem
schedule_conflict
cost_issue
illness
other

confidence must be between 0 and 1.

summary must be less than 25 words.

recommended_action must be short and practical.
"""


def analyse_note(note_text):

    prompt = f"""
    Analyse the following clinic no-show note.

    Note:
    {note_text}
    """

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            temperature=0,
            max_output_tokens=1000,
            response_mime_type="application/json",
            response_schema=SCHEMA
        )
    )

    return json.loads(response.text)


In [76]:
N001 = """
Patient forgot the appointment and did not notice the reminder message.
"""

result = analyse_note(N001)

print(result)

{'root_cause': 'forgot_or_no_reminder', 'confidence': 0.98, 'summary': 'The patient missed the visit because they forgot the appointment and overlooked the reminder message.', 'recommended_action': 'Implement multi-channel reminders, such as automated phone calls alongside SMS alerts.'}


In [81]:
#Task4.2
import pandas as pd

notes = pd.read_csv("followup_notes.csv")

print(notes.columns.tolist())
print(notes.head())

['note_id', 'appointment_id', 'note_text']
  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [83]:
import pandas as pd
import time

from pydantic import BaseModel, Field, ValidationError

# ---------------------------------------
# Allowed root causes
# ---------------------------------------

ALLOWED_ROOT_CAUSES = {
    "forgot_or_no_reminder",
    "transport_problem",
    "schedule_conflict",
    "cost_issue",
    "illness",
    "other"
}

# ---------------------------------------
# Pydantic model
# ---------------------------------------

class FollowUpInsight(BaseModel):
    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

# ---------------------------------------
# Read CSV
# ---------------------------------------

notes = pd.read_csv("followup_notes.csv")

results = []

# ---------------------------------------
# Process first 10 rows
# ---------------------------------------

for _, row in notes.head(10).iterrows():

    note_id = row["note_id"]
    appointment_id = row["appointment_id"]
    note_text = row["note_text"]

    try:

        # -----------------------------------
        # Blank note handling (N004)
        # -----------------------------------

        if pd.isna(note_text) or str(note_text).strip() == "":

            results.append({
                "note_id": note_id,
                "appointment_id": appointment_id,
                "root_cause": "other",
                "confidence": 0.0,
                "summary": "No note recorded",
                "recommended_action": "Re-call patient",
                "validation_status": "skipped_blank",
                "raw_response": ""
            })

            continue

        # -----------------------------------
        # Gemini call
        # -----------------------------------

        reply = analyse_note(note_text)

        raw_response = str(reply)

        try:

            validated = FollowUpInsight(**reply)

            if validated.root_cause in ALLOWED_ROOT_CAUSES:
                validation_status = "valid"
            else:
                validation_status = "invalid"

        except ValidationError:

            validation_status = "invalid"

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": reply.get("root_cause", ""),
            "confidence": reply.get("confidence", ""),
            "summary": reply.get("summary", ""),
            "recommended_action": reply.get("recommended_action", ""),
            "validation_status": validation_status,
            "raw_response": raw_response
        })

        # tier-rate protection
        time.sleep(1)

    except Exception as e:

        results.append({
            "note_id": note_id,
            "appointment_id": appointment_id,
            "root_cause": "",
            "confidence": "",
            "summary": "",
            "recommended_action": "",
            "validation_status": "invalid",
            "raw_response": str(e)
        })

# ---------------------------------------
# Save CSV
# ---------------------------------------

notes_analysed = pd.DataFrame(results)

notes_analysed.to_csv(
    "notes_analysed.csv",
    index=False
)

print(notes_analysed)

print("\nShape:", notes_analysed.shape)
print("\nFile saved: notes_analysed.csv")

  note_id appointment_id         root_cause confidence  \
0    N001          A0872                                 
1    N002          A0897  transport_problem       0.95   
2    N003          A0526         cost_issue       0.98   
3    N004          A0926              other        0.0   
4    N005          A0534                                 
5    N006          A0507  schedule_conflict       0.98   
6    N007          A0333                                 
7    N008          A0222  transport_problem       0.98   
8    N009          A0977                                 
9    N010          A0596  schedule_conflict       0.95   

                                             summary  \
0                                                      
1  Patient was unable to arrange transportation a...   
2  Patient skipped the visit due to high consulta...   
3                                   No note recorded   
4                                                      
5  Patient missed appoint